# Palm Fruit Ripeness Detector — Training YOLOv8 (Classification)

Notebook ini untuk dijalankan di **Google Colab** (butuh GPU: Runtime > Change runtime type > GPU).

**Dataset:** [ripeness-of-oil-palm-fruit](https://www.kaggle.com/datasets/ramadanizikri112/ripeness-of-oil-palm-fruit) — 3.000 gambar (1.000 per kelas: unripe, ripe, overripe), 224x224px.

Catatan: dataset ini formatnya klasifikasi (1 foto = 1 label utuh), bukan object detection dengan bounding box. Jadi kita pakai **YOLOv8 classification mode** (`yolov8n-cls`), bukan detection mode — lebih simpel dan pas untuk struktur data seperti ini.

Alur:
1. Install YOLOv8 (ultralytics)
2. Download dataset dari Kaggle
3. Split & susun ulang folder ke format train/val
4. Training model classification
5. Export ke format ONNX
6. Download hasil model, taruh di `backend/model/palm_ripeness.onnx`

In [ ]:
!pip install ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.4 MB/s eta 0:00:00


## 1. Download Dataset dari Kaggle

Cara download pakai Kaggle API:
1. Buka kaggle.com > Account > Create New API Token (download `kaggle.json`)
2. Upload `kaggle.json` ke Colab lewat cell di bawah

In [ ]:
from google.colab import files
print('Upload file kaggle.json kamu:')
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Upload file kaggle.json kamu:


Saving kaggle.json to kaggle.json


In [ ]:
!kaggle datasets download -d ramadanizikri112/ripeness-of-oil-palm-fruit
!unzip -q *.zip -d raw_dataset

# Cek struktur foldernya dulu, biasanya berupa folder per kelas
!find raw_dataset -maxdepth 3 -type d

Dataset URL: https://www.kaggle.com/datasets/ramadanizikri112/ripeness-of-oil-palm-fruit
License(s): unknown
100% 926M/926M [00:24<00:00, 39.3MB/s]

raw_dataset
raw_dataset/Terlalu Masak
raw_dataset/Masak
raw_dataset/Belum Masak


## 2. Split Dataset ke Train/Val

YOLOv8 classification butuh struktur folder:
```
dataset/
  train/
    unripe/*.jpg
    ripe/*.jpg
    overripe/*.jpg
  val/
    unripe/*.jpg
    ripe/*.jpg
    overripe/*.jpg
```

**Penting:** cek dulu hasil `find raw_dataset ...` di atas — sesuaikan `SOURCE_DIR` dan nama folder kelas (`CLASS_NAMES`) di bawah dengan struktur asli hasil unzip (kadang ada folder pembungkus tambahan, atau nama kelasnya beda kapitalisasi).

In [ ]:
import os, random, shutil

SOURCE_DIR = 'raw_dataset'

# Nama folder ASLI di dataset (Bahasa Indonesia) -> nama kelas versi INGGRIS yang dipakai training
FOLDER_TO_CLASS = {
    'Belum Masak': 'unripe',
    'Masak': 'ripe',
    'Terlalu Masak': 'overripe',
}

VAL_SPLIT = 0.2  # 20% data buat validasi

random.seed(42)

for cls in FOLDER_TO_CLASS.values():
    for split in ['train', 'val']:
        os.makedirs(f'dataset/{split}/{cls}', exist_ok=True)

for src_name, cls in FOLDER_TO_CLASS.items():
    # Cari folder ini di dalam SOURCE_DIR (bisa nested)
    matches = []
    for root, dirs, _ in os.walk(SOURCE_DIR):
        for d in dirs:
            if d.strip().lower() == src_name.strip().lower():
                matches.append(os.path.join(root, d))
    if not matches:
        print(f'⚠️  Folder "{src_name}" tidak ditemukan, cek ulang FOLDER_TO_CLASS/SOURCE_DIR')
        continue

    src_folder = matches[0]
    images = [f for f in os.listdir(src_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.shuffle(images)

    n_val = int(len(images) * VAL_SPLIT)
    val_images = images[:n_val]
    train_images = images[n_val:]

    for img in train_images:
        shutil.copy(os.path.join(src_folder, img), f'dataset/train/{cls}/{img}')
    for img in val_images:
        shutil.copy(os.path.join(src_folder, img), f'dataset/val/{cls}/{img}')

    print(f'{src_name} -> {cls}: {len(train_images)} train, {len(val_images)} val')

Belum Masak -> unripe: 368 train, 92 val
Masak -> ripe: 368 train, 92 val
Terlalu Masak -> overripe: 368 train, 92 val


## 3. Training

Pakai model `yolov8n-cls` (nano classification) — paling ringan & cepat, cocok untuk portofolio/demo. Dataset cuma 3.000 gambar 224x224, jadi training bakal jauh lebih cepat dibanding object detection (~15-30 menit di GPU T4 gratis Colab untuk 50 epoch).

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n-cls.pt')  # pretrained nano classification model

results = model.train(
    data='dataset',  # otomatis detect folder train/ dan val/ di dalamnya
    epochs=50,
    imgsz=224,  # samakan dengan ukuran asli gambar dataset
    batch=32,
    name='palm_ripeness'
)

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.118 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fr

## 4. Evaluasi Cepat

In [7]:
metrics = model.val()
print(metrics)

# Sebagai perbandingan: riset dengan dataset yang sama (InceptionV3 + ANN)
# mendapat overall accuracy 75.94%. Model YOLOv8-cls kita ditargetkan
# minimal setara atau lebih baik dari itu.

Ultralytics 8.4.118 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLOv8n-cls summary (fused): 30 layers, 1,438,723 parameters, 0 gradients, 3.3 GFLOPs
train: /content/dataset/train... found 1104 images in 3 classes ✅ 
val: /content/dataset/val... found 276 images in 3 classes ✅ 
test: None...
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 879.4±257.7 MB/s, size: 66.9 KB)
val: Scanning /content/dataset/val... 276 images, 0 corrupt: 100% ━━━━━━━━━━━━ 276/276 30.5Mit/s 0.0s
val: /content/dataset/val/overripe/Terlalu Masak (46).jpg: corrupt JPEG restored and saved
val: /content/dataset/val/unripe/Belum Masak (123).jpg: corrupt JPEG restored and saved
val: /content/dataset/val/unripe/Belum Masak (223).jpg: corrupt JPEG restored and saved
val: /content/dataset/val/unripe/Belum Masak (334).jpg: corrupt JPEG restored and saved
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 18/18 1.4it/s 12.5s
                   all      0.982          1
Speed: 0.0ms prep

## 5. Export ke ONNX

Ini format yang dipakai backend Node.js kita (`onnxruntime-node`).

In [8]:
model.export(format='onnx', imgsz=224)

import shutil
shutil.copy('runs/classify/palm_ripeness/weights/best.onnx', 'palm_ripeness.onnx')

print('Selesai! File palm_ripeness.onnx siap didownload.')

Ultralytics 8.4.118 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from '/content/runs/classify/palm_ripeness/weights/best.pt' with input shape (1, 3, 224, 224) BCHW and output shape(s) (1, 3) (2.8 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 335ms
Prepared 4 packages in 5.14s
Installed 4 packages in 506ms
 + colorama==0.4.6
 + onnx==1.22.0
 + onnxruntime==1.28.0
 + onnxslim==0.1.95

requirements: AutoUpdate success ✅ 6.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 20...
ONNX: slimming with onnxslim 0.1.95...
ONNX: export success ✅ 8.0s, saved as '/content/runs/class

In [9]:
from google.colab import files
files.download('palm_ripeness.onnx')

# Setelah terdownload, taruh file ini di: backend/model/palm_ripeness.onnx

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>